In [1]:
# Run this cell only once
import sys
import os
os.chdir("../..")
print(os.getcwd())
sys.path.insert(1, './code')

/aloy/home/mgarciac/PocketVec/Code_Article/PocketVec


In [4]:
from utils.pocketvec_utils import prepare_pdb_moe
from utils.pocketvec_utils import prepare_pdb_pdbfixer
from utils.pocketvec_utils import select_first_model
from utils.pocketvec_utils import select_chain
from utils.pocketvec_utils import remove_ligands
from utils.pocketvec_utils import remove_hydrogens
from utils.pocketvec_utils import select_occupancies
from utils.pocketvec_utils import extract_ligand_coords
from utils.pocketvec_utils import calculate_centroid
from utils.pocketvec_utils import create_pocket_centroid
from utils.pocketvec_utils import create_fpocket_ds
from utils.pocketvec_utils import read_fpocket_centers
from utils.pocketvec_utils import read_fpocket_scores
from utils.pocketvec_utils import read_prank_scores
import urllib.request
from Bio.PDB import *
import pandas as pd
import numpy as np
import pickle
import shutil
import pybel
import sys
import os

## This notebook exemplifies how to **download**, **process** and **prepare** protein structures and pockets before feeding them into the PocketVec pipeline to generate binding site descriptors. The notebook is divided in 3 sections:

In [5]:
outpath = "examples/1_preparation/"

# PDB CODE - HAS PDB FILE
# pdb_code = '1A42'
# is_mmcif = False
# file_extension = ".pdb"
# chain = 'A'
# ligand = 'BZU'

# PDB CODE - ONLY HAS MMCIF FILE
pdb_code = "8WO1"
is_mmcif = True
file_extension = ".cif"
chain = 'C'
ligand = "0IL"

### 1. Protein structure

##### DOWNLOAD #####

In [6]:
# Download pdb file
download = urllib.request.urlretrieve('http://files.rcsb.org/download/' + pdb_code + file_extension, os.path.join(outpath, pdb_code + file_extension))

##### PROCESS #####

In [7]:
# 1. Select the first model
path_in = os.path.join(outpath, pdb_code + file_extension)
path_out = os.path.join(outpath, pdb_code + "_model" + file_extension)
select_first_model(path_in, path_out, is_mmcif)

In [8]:
# 2. Select the specified chain
path_in = path_out
path_out = os.path.join(outpath, pdb_code + "_" + chain + ".pdb")
select_chain(path_in, path_out, chain, is_mmcif)

In [9]:
# 3. Remove ligands and waters
path_in = path_out
path_out = os.path.join(outpath, pdb_code + "_ligands.pdb")
ligands = ['HOH', 'BZU', 'HG', 'ZN', '2IL', 'A1L', 'NAG', 'X6N', '0IL']
remove_ligands(path_in, path_out, ligands)

In [10]:
# For further processing, we recommend to use the structure_cheking utility from BioBB. You can
# download it from https://github.com/bioexcel/biobb_structure_checking. Change the following
# variable according to your local path 
path_to_check_structure = '/aloy/home/acomajuncosa/programs/structureChecking/bin/check_structure'

# 4. Remove hydrogens
path_in = path_out
path_out = os.path.join(outpath, pdb_code + "_hydrogens.pdb")
logs1 = remove_hydrogens(path_to_check_structure, path_in, path_out)

# 5. Select occupancies
path_in = path_out
path_out = os.path.join(outpath, pdb_code + "_alt.pdb")
logs2 = select_occupancies(path_to_check_structure, path_in, path_out)

/aloy/home/mgarciac/anaconda3/envs/pocketvec_env/lib/python3.7/site-packages/Bio/pairwise2.py:283: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  BiopythonDeprecationWarning,
/aloy/home/mgarciac/anaconda3/envs/pocketvec_env/lib/python3.7/site-packages/Bio/pairwise2.py:283: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  BiopythonDeprecationWarning,


##### PREPARE (Should take no more than ~1min for normal sized pdbs)

In [11]:
infile = os.path.join(outpath, pdb_code + "_alt.pdb")
outfile = os.path.join(outpath, pdb_code + "_prepared.mol2")
logfile = os.path.join(outpath, "prep_logs.log")

# prepare_pdb_moe(infile, outfile, logfile)
prepare_pdb_pdbfixer(infile, outfile, logfile)

Preparation completed

### 2. Pocket centroid (ligand-based)

In [10]:
# 1. Extract ligand coordinates
path_to_st = "examples/1_preparation/" + pdb_code + ".pdb"
coords = extract_ligand_coords(path_to_st, ligand, chain)

In [11]:
# 2. Calculate pocket centroid
centroid = calculate_centroid(coords)

In [12]:
# 3. Create pocket centroid
path_out = os.path.join(outpath, "CTR_LIG.sd")
create_pocket_centroid(centroid, path_out)

1 molecule converted


### 3. Prepare pocket centroid (pocket-detection)

In [13]:
outpath = "examples/1_preparation/"

# To predict pocket locations, we recommend the combined use of fpocket detection and
# prank scoring. You can download the software from https://github.com/Discngine/fpocket
# and https://github.com/rdk/p2rank. Change the following variables according to your local paths

path_to_fpocket = "/aloy/home/acomajuncosa/programs/fpocket/bin/fpocket"
path_to_prank = "/aloy/home/acomajuncosa/programs/p2rank-2.4/distro/prank"

In [14]:
# 1. Change format from mol2 to pdb
path_to_mol2 = os.path.join(outpath, pdb_code + "_prepared.mol2")
path_to_pdb = os.path.join(outpath, pdb_code + "_prepared.pdb")

command = " ".join(['obabel', path_to_mol2, '-O', path_to_pdb])
os.system(command)

*** Open Babel Warning  in ReadMolecule
  Failed to kekulize aromatic bonds in MOL2 file (title is 1A42_alt.A)

1 molecule converted


0

In [15]:
# 2. Change working directory and run fpocket
os.chdir(outpath)
command = " ".join([path_to_fpocket, '-f', pdb_code + "_prepared.pdb"])
out = os.system(command)
os.chdir("../..")

***** POCKET HUNTING BEGINS ***** 
***** POCKET HUNTING ENDS ***** 


mkdir: cannot create directory ‘1A42_prepared_out/pockets’: File exists


In [17]:
# 3. Rerank with PRANK
path_to_dsfile = "examples/1_preparation/fpocket3.ds"
create_fpocket_ds(pdb_code, path_to_dsfile)
command = " ".join([path_to_prank, 'rescore', path_to_dsfile, '-o', 'examples/1_preparation/prank'])
os.system(command)

----------------------------------------------------------------------------------------------
 P2Rank 2.4-beta.2
----------------------------------------------------------------------------------------------

rescoring pockets on proteins from dataset [fpocket3.ds]
processing [1A42_prepared_out.pdb] (1/1)
rescoring finished in 0 hours 0 minutes 4.159 seconds
results saved to directory [/aloy/home/acomajuncosa/PocketVec_v2/GitLab_repo/examples/1_preparation/prank]

----------------------------------------------------------------------------------------------
 finished successfully in 0 hours 0 minutes 5.218 seconds
----------------------------------------------------------------------------------------------


0

In [18]:
# 4. Read fpocket & prank results & calculate pocket centroids
fpockets = read_fpocket_scores(os.path.join(outpath, pdb_code + "_prepared_out", pdb_code + "_prepared_info.txt"))
centers = read_fpocket_centers(os.path.join(outpath, pdb_code + "_prepared_out", pdb_code + "_prepared_pockets.pqr"))
centers = {i: calculate_centroid(centers[i]) for i in sorted(centers)}
pranks = read_prank_scores(os.path.join(outpath, 'prank', pdb_code + "_prepared_out.pdb_rescored.csv"))

In [19]:
# 5. In dicts fpockets (and pranks) we have pocket scores (re-scores) provided by fpocket (prank)
print("FPOCKET Druggability Scores")
print("\n".join([str(i) + " --> " + str(fpockets[i]['Druggability Score']) for i in sorted(fpockets)]))
print("\n\n")
print("PRANK Re-scores")
print("\n".join([str(i) + " --> " + str(pranks[i]) for i in sorted(fpockets)]))

FPOCKET Druggability Scores
1 --> 0.649
2 --> 0.337
3 --> 0.0
4 --> 0.025
5 --> 0.001
6 --> 0.009
7 --> 0.002
8 --> 0.046
9 --> 0.013
10 --> 0.001
11 --> 0.0
12 --> 0.001
13 --> 0.0
14 --> 0.001
15 --> 0.0
16 --> 0.0



PRANK Re-scores
1 --> 12.6436
2 --> 30.1467
3 --> 1.3809
4 --> 1.0933
5 --> 28.399
6 --> 2.3902
7 --> 0.6167
8 --> 9.5715
9 --> 3.2372
10 --> 2.1658
11 --> 2.7282
12 --> 1.3517
13 --> 1.0688
14 --> 0.8161
15 --> 0.9472
16 --> 0.7296


In [20]:
# 6. Print pocket centroids (pocket detection, PD)

if os.path.exists(os.path.join(outpath, "pockets_PD")) == False: os.makedirs(os.path.join(outpath, "pockets_PD"))

for pocket in centers:
    centroid = centers[pocket]
    outfile = os.path.join(outpath, "pockets_PD", "CTR_PD_" + str(pocket) + ".sd")
    create_pocket_centroid(centroid, outfile)

1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
